In [3]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [6]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model="gpt-4.1-nano")

In [7]:
# Creating custom tools with Langchain
## Defining an add function
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): First integer to be added
        b (int): second interger to be added
        
    Return:
        int: sum of a and b
    """
    
    return a + b

In [8]:
tools = [add]

llm_with_tools = llm.bind_tools(tools)

In [9]:
# Subtract tool
def subtract(a: int, b: int) -> int:
    """
        Subtract b from a.
    """
    return a - b

In [10]:
# Multiply tool 
def multiply(a: int, b: int) -> int:
    """
        Multiply a and b.
    """
    return a * b

In [11]:
# Testing the functions
tool_map = {
    "add": add, 
    "subtract": subtract,
    "multiply": multiply
}

input_ = {
    "a": 1,
    "b": 2
}

tool_map["add"].invoke(input_)

3

In [12]:
tools = [add, subtract, multiply]

llm_with_tools = llm.bind_tools(tools)

In [13]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

In [14]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

In [15]:
print(type(response_1))
print(response_1)

<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'tool_calls': [{'id': 'call_EGthwvuxmwITy6eFD7I3hflu', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'add'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 133, 'total_tokens': 150, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--019c65fc-77e5-7512-a913-36591486ec30-0' tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 2}, 'id': 'call_EGthwvuxmwITy6eFD7I3hflu', 'type': 'tool_call'}] usage_metadata={'input_tokens': 133, 'output_tokens': 17, 'total_tokens': 150, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0,

In [16]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name:\n{tool_1_name}')
print(f'tool args:\n{tool_1_args}')
print(f'tool call ID:\n{tool_call_1_id}')

tool name:
add
tool args:
{'a': 3, 'b': 2}
tool call ID:
call_EGthwvuxmwITy6eFD7I3hflu


In [17]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

content='5' tool_call_id='call_EGthwvuxmwITy6eFD7I3hflu'


In [18]:
chat_history.append(tool_message)

In [19]:
answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
3 + 2 equals 5.
